<a href="https://colab.research.google.com/github/senanurcetin/APTOS-2019-diabetic-retinopathy/blob/main/aptos_2019.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Environment and libraries

In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

### Google Cloud authentication

In [ ]:
from google.colab import auth

auth.authenticate_user()
print("Authenticated.")

### Shared preprocessing module

In [ ]:
# ==============================================================================
# LOAD THE SHARED PREPROCESSING MODULE
# ==============================================================================
# The pipeline functions live in scripts/preprocessing.py. We import them
# rather than keeping a copy here, so the notebook and the scripts always do
# the same thing and neither falls behind the other.
import os, sys, subprocess

REPO = "/content/aptos-dr"
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/senanurcetin/APTOS-2019-diabetic-retinopathy.git", REPO],
                   check=True)
else:
    subprocess.run(["git", "-C", REPO, "pull", "--quiet"], check=False)

sys.path.insert(0, f"{REPO}/scripts")

from preprocessing import (auto_crop, apply_clahe, to_square, image_quality,
                           preprocess, dhash)

print("shared module loaded:", REPO + "/scripts/preprocessing.py")

### Configuration

In [ ]:
# ==============================================================================
# PROJECT CONFIGURATION - single source of truth for the cells below
# ==============================================================================
PROJECT_ID  = "datascientis"
BUCKET_NAME = "aptos2019-retina-images"
BQ_DATASET  = "APTOS_2019"

LOCAL_DIR = "/content/aptos_images"   # local image cache

BQ_TABLES = {
    "train": f"{PROJECT_ID}.{BQ_DATASET}.aptos_train",
    "valid": f"{PROJECT_ID}.{BQ_DATASET}.aptos_valid",
    "test":  f"{PROJECT_ID}.{BQ_DATASET}.aptos_test",
}


def blob_name(image_uri):
    """gs://bucket/folder/file.png  ->  folder/file.png"""
    return image_uri.replace(f"gs://{BUCKET_NAME}/", "", 1)


def local_path(image_uri):
    return f"{LOCAL_DIR}/{blob_name(image_uri)}"


print("Configuration ready.")

### Reading the tables from BigQuery

In [ ]:
from google.cloud import bigquery

bq_client = bigquery.Client(project=PROJECT_ID)


# The image_uri column carries the full gs:// path, including the folder
# prefix (aptos_train_images/ and so on), so there is nothing to assemble.
def fetch(split):
    q = f"SELECT id_code, diagnosis, image_file, image_uri FROM `{BQ_TABLES[split]}`"
    return bq_client.query(q).to_dataframe()


df_train, df_valid, df_test = fetch("train"), fetch("valid"), fetch("test")

for name, df in [("Train", df_train), ("Valid", df_valid), ("Test", df_test)]:
    print(f"{name:<6} {len(df):>5} rows")

print("\nexample image_uri:", df_train["image_uri"].iloc[0])

### Caching the images on local disk

In [ ]:
# ==============================================================================
# CACHE THE IMAGES ON LOCAL DISK  (runs once, ~8 GB)
# ==============================================================================
# Why: fetching from GCS inside every __getitem__ runs at ~2.7 images/second,
# so a 2930-image epoch would spend ~18 minutes just reading. Copying once to
# local disk keeps training off the I/O bottleneck.
import os

os.makedirs(LOCAL_DIR, exist_ok=True)
!gsutil -m -q cp -r "gs://$BUCKET_NAME/*" "$LOCAL_DIR/"

for split, df in [("train", df_train), ("valid", df_valid), ("test", df_test)]:
    have = sum(os.path.exists(local_path(u)) for u in df["image_uri"])
    print(f"{split:<6} {have}/{len(df)} images cached locally")

### Verifying the paths

In [ ]:
# Confirm the image_uri paths really resolve in the bucket
from google.cloud import storage

bucket = storage.Client(project=PROJECT_ID).bucket(BUCKET_NAME)

for uri in df_train["image_uri"].head(3):
    status = "FOUND" if bucket.blob(blob_name(uri)).exists() else "MISSING"
    print(f"{uri}\n   -> {status}")

### Dataset class

In [ ]:
import io
import os

import numpy as np
import torch
import torchvision.transforms as T
from PIL import Image
from torch.utils.data import DataLoader, Dataset


class APTOSDataset(Dataset):
    """Reads from the local cache, falling back to GCS.

    The preprocessing steps come from the shared module (auto_crop ->
    apply_clahe -> to_square); the notebook keeps no copy of its own.
    """

    def __init__(self, df, transform=None, use_clahe=True, size=512):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.use_clahe = use_clahe
        self.size = size
        self._bucket = None

    def __len__(self):
        return len(self.df)

    def read_bytes(self, image_uri):
        path = local_path(image_uri)
        if os.path.exists(path):
            with open(path, "rb") as fh:
                return fh.read()
        if self._bucket is None:
            from google.cloud import storage
            self._bucket = storage.Client(project=PROJECT_ID).bucket(BUCKET_NAME)
        return self._bucket.blob(blob_name(image_uri)).download_as_bytes()

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        pil = Image.open(io.BytesIO(self.read_bytes(row["image_uri"]))).convert("RGB")

        # PIL(RGB) -> OpenCV(BGR), shared module, then back again
        bgr = np.array(pil)[:, :, ::-1].copy()
        bgr = auto_crop(bgr)
        if self.use_clahe:
            bgr = apply_clahe(bgr)
        bgr = to_square(bgr, self.size, mode="squash")
        img = Image.fromarray(bgr[:, :, ::-1])

        if self.transform:
            img = self.transform(img)

        return img, torch.tensor(int(row["diagnosis"]), dtype=torch.long)

### Dataframes and transforms

In [ ]:
train_transform = T.Compose([
    T.Resize((224, 224)),
    T.RandomHorizontalFlip(p=0.5),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

eval_transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

train_dataset = APTOSDataset(df_train, transform=train_transform)
valid_dataset = APTOSDataset(df_valid, transform=eval_transform)
test_dataset  = APTOSDataset(df_test,  transform=eval_transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True,
                          num_workers=2, pin_memory=True)
valid_loader = DataLoader(valid_dataset, batch_size=32, shuffle=False,
                          num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=32, shuffle=False,
                          num_workers=2, pin_memory=True)

imgs, labels = next(iter(train_loader))
print("batch:", imgs.shape, labels.shape)

# Exploratory data analysis

### Class distribution and imbalance ratios

In [ ]:
# Class distribution
# Class names
class_names = {
    0: "No DR",
    1: "Mild",
    2: "Moderate",
    3: "Severe",
    4: "Proliferative DR"
}

datasets = {
    "Train": df_train,
    "Validation": df_valid,
    "Test": df_test
}

for dataset_name, df in datasets.items():

    # Images per class
    counts = (
        df["diagnosis"]
        .value_counts()
        .reindex(range(5), fill_value=0)
        .sort_index()
    )

    # Percentage share
    percentages = (counts / counts.sum()) * 100

    # Summary table
    distribution_table = pd.DataFrame({
        "Grade": counts.index,
        "Class": [class_names[i] for i in counts.index],
        "Images": counts.values,
        "Share (%)": percentages.round(2).values
    })

    print(f"\n{dataset_name} class distribution")
    display(distribution_table)

    # Class imbalance
    non_zero_counts = counts[counts > 0]

    max_count = non_zero_counts.max()
    min_count = non_zero_counts.min()

    imbalance_ratio = max_count / min_count

    print(f"Most frequent: grade {non_zero_counts.idxmax()} -> {max_count}")
    print(f"Least frequent: grade {non_zero_counts.idxmin()} -> {min_count}")
    print(f"Imbalance ratio (max / min): {imbalance_ratio:.2f}")

    # Chart
    plt.figure(figsize=(8, 5))

    bars = plt.bar(
        [f"Grade {i}" for i in counts.index],
        counts.values
    )

    plt.title(f"{dataset_name} - class distribution")
    plt.xlabel("Diyabetik Retinopati Evresi")
    plt.ylabel("images")

    for bar, count, percentage in zip(
        bars, counts.values, percentages.values
    ):
        plt.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height(),
            f"{count}\n%{percentage:.1f}",
            ha="center",
            va="bottom"
        )

    plt.tight_layout()
    plt.show()

### Data quality checks

In [ ]:
from PIL import Image, ImageStat
from tqdm.auto import tqdm


def check_dataset_quality(df, brightness_threshold=12.0):
    """Find unreadable and unusually dark images."""
    reader = APTOSDataset(df, use_crop=False)
    corrupted, too_dark, healthy = [], [], []

    for idx in tqdm(range(len(df)), desc="checking images"):
        row = df.iloc[idx]
        try:
            img = Image.open(io.BytesIO(reader.read_bytes(row["image_uri"]))).convert("RGB")
            brightness = ImageStat.Stat(img.convert("L")).mean[0]
            target = too_dark if brightness < brightness_threshold else healthy
            target.append({"id_code": row["id_code"], "brightness": brightness})
        except Exception as e:
            corrupted.append({"id_code": row["id_code"], "error": str(e)})

    print("\n" + "=" * 40)
    print("QUALITY CHECK RESULTS")
    print("=" * 40)
    print(f"Healthy images  : {len(healthy)}")
    print(f"Unreadable      : {len(corrupted)}")
    print(f"Too dark        : {len(too_dark)}")
    return healthy, corrupted, too_dark


healthy_list, corrupted_list, dark_list = check_dataset_quality(df_train)